<a href="https://colab.research.google.com/github/s61711299-sketch/MSSP6070/blob/main/PARTICIPATION_ACTIVITY_Week_9_Fetch_a_website's_content_and_extract_text_from_it.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

In [9]:
url = "https://www.ikea.com/us/en/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

res = requests.get(url, headers=headers, timeout=30)

print("Status code:", res.status_code)
res.raise_for_status()

soup = BeautifulSoup(res.text, "html.parser")

image_urls = []

for img in soup.find_all("img"):
    image_url = (
        img.get("src")
        or img.get("data-src")
        or img.get("data-lazy-src")
    )

    if not image_url and img.get("srcset"):
        image_url = (
            img.get("srcset")
            .split(",")[0]
            .strip()
            .split()[0]
        )

    if image_url:
        image_url = urljoin(url, image_url)

        domain = urlparse(image_url).hostname or ""

        if domain == "ikea.com" or domain.endswith(".ikea.com"):
            if image_url not in image_urls:
                image_urls.append(image_url)

print("Number of image URLs found:", len(image_urls))

for image_url in image_urls[:10]:
    print(image_url)

Status code: 200
Number of image URLs found: 108
https://www.ikea.com/global/assets/logos/brand/ikea.svg
https://www.ikea.com/us/en/range-categorisation/images/product/winter-holiday-wt001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/storage-furniture-st001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/sofas-armchairs-700640.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/beds-mattresses-bm001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/lighting-li001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/kitchen-appliances-ka001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/textiles-rugs-tl001.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/tables-chairs-fu002.jpeg?imwidth=160
https://www.ikea.com/us/en/range-categorisation/images/product/desk-desk-chairs-fu004

In [10]:
save_folder = "ikea_images"
os.makedirs(save_folder, exist_ok=True)

downloaded = 0
max_images = 10

for index, image_url in enumerate(image_urls[:max_images], start=1):
    try:
        img_res = requests.get(
            image_url,
            headers=headers,
            stream=True,
            timeout=30
        )
        img_res.raise_for_status()

        content_type = (
            img_res.headers.get("Content-Type", "")
            .split(";")[0].lower()
        )

        extensions = {
            "image/jpeg": ".jpg",
            "image/png": ".png",
            "image/webp": ".webp",
            "image/gif": ".gif"
        }

        extension = extensions.get(content_type)

        if extension is None:
            print("Skipped non-image:", image_url)
            img_res.close()
            continue

        filepath = os.path.join(
            save_folder, f"ikea_{index}{extension}"
        )

        with open(filepath, "wb") as image_file:
            for chunk in img_res.iter_content(100000):
                if chunk:
                    image_file.write(chunk)

        img_res.close()
        downloaded += 1
        print("Downloaded:", filepath)

    except requests.RequestException as error:
        print("Download failed:", error)

print("Total images downloaded:", downloaded)

Skipped non-image: https://www.ikea.com/global/assets/logos/brand/ikea.svg
Downloaded: ikea_images/ikea_2.jpg
Downloaded: ikea_images/ikea_3.jpg
Downloaded: ikea_images/ikea_4.jpg
Downloaded: ikea_images/ikea_5.jpg
Downloaded: ikea_images/ikea_6.jpg
Downloaded: ikea_images/ikea_7.jpg
Downloaded: ikea_images/ikea_8.jpg
Downloaded: ikea_images/ikea_9.jpg
Downloaded: ikea_images/ikea_10.jpg
Total images downloaded: 9


In [11]:
import shutil
import os
from google.colab import files

folder = "/content/ikea_images"

if os.path.exists(folder) and os.listdir(folder):
    zip_path = shutil.make_archive(
        "/content/ikea_images",
        "zip",
        folder
    )
    files.download(zip_path)
else:
    print("empty")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>